# Lab 1 古典 IR：親手做出一個最小搜尋引擎（教學版）

**今天的目標：** 親手做出一個會「打分數、排名次」的最小搜尋引擎——輸入查詢，回傳排序後最相關的文件與分數。它就是下午 RAG「檢索」那一半的原型。

**這本是教學版：** 每格的核心程式碼挖成 `____`，照 `# TODO` 提示自己填。回家可對照**完整版**（含全部解答與小作業參考解）。

> 語料為**虛構、教學用**財經新聞（非真實行情）。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（指定版本，避免學校電腦裝到不相容的舊版；裝不起來看 README）
!pip install -q scikit-learn==1.6.1 numpy==1.26.4 pandas==2.2.3 jieba==0.42.1 rank_bm25==0.2.2 matplotlib==3.10.0

## 🔧 第 0 步：環境檢查

先跑下一格，全綠再往下。

In [ ]:
# ✅ 環境檢查：跑這格，全部通過再往下（練 A–C 只需要 sklearn / numpy）
import sys
print("Python：", sys.version.split()[0])

import sklearn, numpy
print("scikit-learn：", sklearn.__version__)
print("numpy：", numpy.__version__)

import glob
news_files = sorted(glob.glob("data/news/*.txt"))
n_files = len(news_files)
print(f"data/news/ 語料：找到 {n_files} 篇（預期 15 篇，練 D 才用到）")

if n_files == 15:
    print("")
    print("✅ 環境 OK，可以開始！（jieba 到練 D 開頭才檢查，現在不用管）")
else:
    print("")
    print("❌ 沒找到 15 篇語料——請確認你是在 Lab1_古典IR/ 資料夾底下開這本 notebook，")
    print("   關掉 Jupyter、切到正確資料夾再重開一次。")

### 🛠 先載入今天畫圖／列表的工具

`matplotlib` 畫散點圖、`pandas` 把資料列成表格。設定一次中文字型，後面畫圖都通用。

In [ ]:
# 畫圖 / 表格工具：整本共用，載一次就好
import matplotlib.pyplot as plt
import pandas as pd
# 這兩行設定中文字型——找不到就用預設，圖照畫、只是中文變方框，不影響看點的位置
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans CJK TC", "Noto Sans CJK SC", "Microsoft JhengHei", "PingFang TC", "AR PL UMing CN", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

### 🎯 向量：一排數字＝一個點

**向量**＝一排數字＝座標。兩個數字就能當平面上一個點，**近＝像**。先用幾部動漫看 👇

In [ ]:
# 每部作品用一個 dict 描述（anime 名稱、hot 熱血、heal 療癒），組成一張表
# 一「列」＝一部作品、一「欄」＝一個軸——這張表就是 6 個「向量」排在一起
works = [
    {"anime": "鬼滅",       "hot": 5, "heal": 1},
    {"anime": "航海王",     "hot": 5, "heal": 2},
    {"anime": "吉伊卡哇",   "hot": 1, "heal": 5},
    {"anime": "芙莉蓮",     "hot": 2, "heal": 4},
    {"anime": "MyGO!!!!!",  "hot": 2, "heal": 2},
    {"anime": "Ave Mujica", "hot": 3, "heal": 1},
]
df_anime = pd.DataFrame(works)
df_anime   # 最後一行不寫 print，Jupyter 直接把表格畫出來

In [ ]:
# 把上面那張表的每一列，畫成平面上的一個點（不用填，直接跑）
plt.figure(figsize=(5, 5))
for w in works:                              # 每部作品＝(熱血, 療癒) 兩個數字＝一個點
    plt.scatter(w["hot"], w["heal"], s=150, color="steelblue")
    plt.annotate(w["anime"], (w["hot"], w["heal"]), xytext=(6, 6), textcoords="offset points")

plt.xlabel("熱血度")
plt.ylabel("療癒度")
plt.title("兩個數字＝平面上一個點：近＝像")
plt.grid(True, alpha=0.3)
plt.show()

# 💡 鬼滅、航海王在右下（熱血高），吉伊卡哇、芙莉蓮在左上（療癒高）——「像不像」變成看得見的遠近。
#    但 MyGO!!!!! 和 Ave Mujica 都偏低療癒、位置相近，其實很不一樣——2 個軸不夠、要更多軸才分得開。

### 🎯 關鍵字向量：文件也變成一排數字

文件的軸＝**每個詞**，格子填分數 → 一篇文件變一排數字（一個點）。今天格子值會升級：布林填 0/1、TF-IDF 填加權分數。先用「出現次數」當分數看 👇

In [ ]:
# 把每篇文件，數一數兩個關鍵字各出現幾次——組成一張表（每列＝一篇的關鍵字向量）
demo_docs = ["升息 台股 升息", "原油 庫存", "升息 衝擊 台股"]   # 三篇迷你文件
rows = []
for doc in demo_docs:
    words = doc.split()
    rows.append({"doc": doc, "升息": words.count("升息"), "台股": words.count("台股")})
df_docs = pd.DataFrame(rows)
df_docs   # 一列＝一篇文件的關鍵字向量（升息, 台股）

In [ ]:
# 把每篇文件也畫成點：座標＝(升息次數, 台股次數)（不用填，直接跑）
plt.figure(figsize=(5, 5))
for r in rows:
    plt.scatter(r["升息"], r["台股"], s=150, color="steelblue")
    plt.annotate(r["doc"], (r["升息"], r["台股"]), xytext=(6, 6), textcoords="offset points")

plt.xlabel("「升息」出現次數")
plt.ylabel("「台股」出現次數")
plt.title("文件也變成點：座標＝關鍵字出現次數")
plt.grid(True, alpha=0.3)
plt.show()

# 💡「升息 台股 升息」→(2,1)、「升息 衝擊 台股」→(1,1)、「原油 庫存」→(0,0) 落在原點（沒這兩個詞）。

---
## 🟦 練 A：手刻倒排索引 + 布林查詢

用 `dict` 刻出**倒排索引**（詞 → 出現在哪些文件），再做布林 AND / OR 查詢。

> ⚠️ 「倒排」的「排」指**方向**倒過來（詞→文件），**跟排名次無關**——排序是練 B 的事。

### A1・準備語料（直接跑，不用改）

5 篇虛構財經新聞，已用空白斷好詞。

In [ ]:
# 5 篇【虛構・教學用】財經新聞短句（非真實行情，僅供教學）
# 用空白把詞切開，方便最小版直接用 split() 斷詞（中文真實斷詞見練 D）
docs = [
    "升息 抑制 通膨 央行 升息 一碼",          # 文件0：升息
    "台股 開高 半導體 領漲 台股 收紅",        # 文件1：台股/半導體
    "原油 庫存 下滑 油價 走高",               # 文件2：原油
    "半導體 毛利率 提升 晶圓代工 報價 上揚",   # 文件3：半導體財報
    "升息 衝擊 台股 資金 外流",               # 文件4：升息+台股
]

for i, d in ____(docs):        # TODO：要「一邊跑清單、一邊拿到編號 (0,1,2…)」——哪個內建函式？
    print(f"文件{i}: {d}")

### A2-1・先看一眼：一句話怎麼變成「不重複的詞」（三步）

建索引之前，先把**單一句話**拆清楚：① 從清單取一句 → ② 切成詞 → ③ 去重。

**預期輸出：** 原文一行；一串詞；最後一組**沒有重複**的詞（文件 0 的「升息」兩次，去重後只剩一個）。

In [ ]:
# ① 先從 docs 這個 list 取出第 0 篇（list 用「位置」取值）
text = docs[____]        # TODO：想取「第 0 篇」，中括號裡放什麼？
print("原文：", text)

In [ ]:
# ② 把句子「用空白切成一串詞」
text.____()    # TODO：字串用哪個方法「按空白切成清單」？（切完你會看到「升息」出現兩次）

In [ ]:
# ③ 去掉重複——倒排索引只在乎「有沒有這個詞」，出現幾次是之後 TF-IDF 的事
set(text.____())    # TODO：先切成詞（同上一格），外面已用 set() 幫你去重了

### A2-2・先認識兩個容器：`list` 與 `dict`

倒排索引就是拿這兩個搭出來的，先各玩一下：`list` 用 `append` 加東西、`dict` 用 `get` 取值。

**預期輸出：** `list append：[0, 4]`；`get（有鍵）`回 `[0, 4]`；`get（沒鍵給預設）`回 `[]`；`get（沒鍵沒預設）`回 `None`。

In [ ]:
# list：一排東西，用 append 加到尾巴
ids = []
ids.append(0)
ids.append(4)
print("list append：", ids)                          # [0, 4]

# dict：用「鍵」查「值」，用 get 取值（查不到不會報錯）
postings = {"升息": [0, 4]}
print("get（有鍵）      ：", postings.get("升息"))         # [0, 4]
print("get（沒鍵給預設）：", postings.get("黃金", []))      # []   ← 給了預設值，回預設
print("get（沒鍵沒預設）：", postings.get("黃金"))          # None ← 沒給預設，回 None（不報錯）
# 💡 下一格建倒排索引就會用到：get(w, []) ＝「查不到就當空清單」，省掉一堆 if 判斷

### A2・手刻倒排索引

建出 `{詞: [文件編號]}`。

**預期輸出：** `升息 → [0, 4]`　`台股 → [1, 4]`　`半導體 → [1, 3]`　`原油 → [2]`

In [ ]:
inverted = {}                      # 倒排索引：{詞: 含此詞的文件編號清單}
for doc_id, text in enumerate(docs):
    for word in set(text.split()): # 上面剛看過：切詞 + 去重
        if word not in inverted:   # 這個詞第一次出現 → 先給它一個空清單
            inverted[word] = []
        inverted[word].____(doc_id)   # TODO：把 doc_id 加到這個詞的清單尾巴——list 的哪個方法？

for w in ["升息", "台股", "半導體", "原油"]:
    print(w, "→", sorted(inverted[w]))

### A3-0・先練一次「集合三兄弟」：交集 / 聯集 / 差集

布林查詢**整個都是集合運算**——先把三個運算單獨玩過一次，等一下才不會卡在語法上。直接拿剛建好的 postings 來玩。

**預期輸出：** `升息 → {0, 4}`、`台股 → {1, 4}`；交集 `{4}`、聯集 `{0, 1, 4}`、差集（升息有台股沒）`{0}`。

In [ ]:
# 集合三兄弟：交集 / 聯集 / 差集——布林查詢的骨架
a = set(inverted[____])   # TODO：哪幾篇有「升息」？中括號裡放那個詞
b = set(inverted[____])   # TODO：哪幾篇有「台股」？

print("升息 →", a)
print("台股 →", b)

In [ ]:
print("交集（兩邊都有）    ：", a ____ b)   # TODO：兩邊「都有」才留下——集合的哪個運算子？（這就是 AND）

In [ ]:
print("聯集（任一邊有）    ：", a ____ b)   # TODO：任一邊「有」就收——哪個運算子？（這就是 OR）

In [ ]:
print("差集（升息有、台股沒）：", a ____ b)  # TODO：升息有、台股沒——「相減」用哪個運算子？（NOT 會用到）
# ⚠️ 三個運算子都只吃「集合」——postings 本體是 list，用之前一定要先 set() 一下

### A3・用倒排索引做布林查詢

`AND` ＝各詞 postings 取**交集**；`OR` ＝取**聯集**。完全不用掃原文。

**預期輸出：** `升息 → [0, 4]`；`台股 → [1, 4]`；`升息 AND 台股 → [4]`；`升息 OR 台股 → [0, 1, 4]`。

> 跑完想一下：它只回「哪些符合」，**沒告訴你哪篇比較相關**——這就是布林的極限。

In [ ]:
# A3-1：boolean_and——AND＝取各詞 postings 的「交集」
def boolean_and(word_list):
    if len(word_list) == 0:
        return []
    result = set(inverted.get(word_list[0], []))   # 第一個詞的 postings，轉集合當起點
    for w in word_list:
        postings = set(inverted.get(w, []))        # 先轉集合再運算（list 不能做 & / |）
        result = result ____ postings              # TODO：AND ＝「兩邊都有」——集合三兄弟的哪一個？
    # get(w, [])：查不存在的詞回空清單而不報錯
    return sorted(result)

In [ ]:
print("台股 →        ", inverted.get("台股", []))

In [ ]:
print("升息 →        ", inverted.get("升息", []))

In [ ]:
print("升息 AND 台股 →", ____(["升息", "台股"]))   # TODO：呼叫剛剛定義的 AND 函式

In [ ]:
# A3-2：boolean_or——OR＝取各詞 postings 的「聯集」
def boolean_or(word_list):
    result = set()                                 # 從空集合開始收
    for w in word_list:
        postings = set(inverted.get(w, []))        # 同 A3-1：先轉集合再運算
        result = result ____ postings              # TODO：OR ＝「任一邊有就收」——哪個運算子？
    return sorted(result)

In [ ]:
print("升息 OR 台股 → ", ____(["升息", "台股"]))   # TODO：呼叫剛剛定義的 OR 函式

### 📝 小作業 A

1. **先在紙上預測、再跑 code 驗證：**「半導體 AND 台股」會回哪幾篇？「原油 OR 半導體」呢？（對照 A2 印出的 postings 表用眼睛算；記得參數是「詞的清單」）
2. **⭐ 進階（選做）：** 寫一個 `boolean_not(word)`，回傳「**不含**該詞」的文件編號。想一想：AND／OR 只要有 postings 就夠，**NOT 需要一個 AND／OR 都不需要的東西**——那是什麼？（提示：NOT 得先知道「總共有哪些文件」；集合三兄弟的第三個 `-` 派上用場，「全部文件的編號」可從 `docs` 的長度生出來。）

<details><summary>📖 做完再看：參考解（參考解不只一種，思路對就好）</summary>

```python
# Q1：先用 postings 表眼睛算——半導體→[1,3]、台股→[1,4] → AND 應只有 1；原油→[2]、半導體→[1,3] → OR 應是 [1,2,3]
print("半導體 AND 台股 →", boolean_and(["半導體", "台股"]))
print("原油 OR 半導體  →", boolean_or(["原油", "半導體"]))

# ⭐ 進階：NOT ＝ 從「全部文件」扣掉含此詞的（全集 - postings）
def boolean_not(word):
    all_ids = set(range(len(docs)))            # 全部文件編號 {0,1,2,3,4}——NOT 才需要的「全集」基準
    has_word = set(inverted.get(word, []))
    return sorted(all_ids - has_word)          # - ＝集合相減
print("NOT 升息 →", boolean_not("升息"))          # [1, 2, 3]
```
</details>

In [ ]:
# 📝 小作業 A —— 你的答案（做完再看上面摺疊參考解對答案）
# Q1：呼叫 boolean_and / boolean_or 驗證你的紙上預測
#     print("半導體 AND 台股 →", ...)
#     print("原油 OR 半導體  →", ...)

# ⭐ 進階：補完 boolean_not（提示：NOT 需要「全集」，用 set(range(len(docs))) 生出來，再集合相減）
# def boolean_not(word):
#     all_ids = ...
#     has_word = ...
#     return sorted(...)


---
## 🟩 練 B：TF-IDF 向量化 + cosine 排序（本 Lab 重點）

把文件變成 **TF-IDF 向量**，查詢也變向量，用 **cosine** 算相似度 → 排序。做完你就有一個會「打分數、排名次」的最小搜尋引擎。

> ⚠️ **sklearn 的分數跟手算對不上是正常的**（它用平滑版 IDF ＋ 正規化）。**不要對數字，看排序。**

### B0・回主軸：格子值從 0/1 升級成加權分數

開場的主軸說過：文件是「關鍵字向量」，布林時格子只填 0/1。這一段 `TfidfVectorizer` 把格子值換成「**這個詞在這篇有多重要**」的加權分數——軸不變、空間不變，只是值變聰明了，才能排出「哪篇最相關」。

### B1・`TfidfVectorizer`：把文件變向量

**預期輸出：** 詞彙表大小 23（三格分別印：詞彙表大小 → 向量矩陣形狀 `(5, 23)` → 前 8 個詞）。

> 🛟 跑出 `(5, 0)` → 漏改 `token_pattern`，中文被吃掉了。

In [ ]:
from sklearn.feature_extraction.text import ____   # TODO：把文件變 TF-IDF 向量的類別（就是本段主角，名字在標題）

# token_pattern：告訴 sklearn「怎麼把一句話切成詞」。
#   預設規則只抓「長度≥2 的英數詞」→ 中文整段被吃掉、shape 變 (5, 0)。
#   r"(?u)\S+" ＝「用空白切，中文也算數」——正好對上我們已用空白切好的語料。
vectorizer = ____(token_pattern=r"(?u)\S+")        # TODO：同一個類別，建一個實例

doc_vectors = vectorizer.____(docs)
# TODO：文件要「先學一份詞彙表＋算 IDF、再轉成向量」——一次做完兩件事的方法是哪個？
#       ⚠️ 全 Lab 最容易錯：文件用「先學再轉」、查詢（B3）用「只轉」。

print("詞彙表大小：", len(vectorizer.get_feature_names_out()))

In [ ]:
print("向量矩陣形狀：", doc_vectors.____, "（5 篇 × 詞彙數）")   # TODO：看矩陣「幾列幾行」的屬性

In [ ]:
print("前 8 個詞：", vectorizer.get_feature_names_out()[____])   # TODO：取清單「前 8 個」的切片寫法

In [ ]:
# 先看一眼：一篇文件長成「一排數字」的樣子
# B1 已把 5 篇文件學成向量（doc_vectors），這裡把「文件0」那一排數字攤開看
print("文件0 原文：", docs[0])
print()
print("文件0 的向量（23 個數字，每一格對應詞彙表裡的一個詞）：")
doc_vectors[0].____()   # TODO：稀疏矩陣「攤成看得見的一排數字」用哪個方法？（B3 還會用到）

### B2・印 IDF：親眼看「罕見詞分數高」（兩格）

**預期輸出：** 「毛利率」「原油」（各只出現 1 篇）IDF ＝ **2.099**；「升息」「台股」「半導體」（各 2 篇）＝ **1.693**。

> **含詞文件越少 → IDF 越大 → 該詞越有鑑別力。**

In [ ]:
# B2-1：先直接看一眼 IDF 的原始長相
vocab = vectorizer.get_feature_names_out()   # fit 之後才有——就是 B1 印「前 8 個詞」用的那個方法

print("詞彙表（23 個詞）：")
print(vocab)
print()
print("每個詞的 IDF（23 個數字，跟上面的詞彙表順序一一對應）：")

In [ ]:
vectorizer.____
# TODO：每個詞的 IDF。sklearn 慣例：fit 之後才學出來的屬性，尾巴帶一個底線，名字＝那三個英文字母小寫＋底線
# 最後一行不寫 print——Jupyter 會直接顯示

In [ ]:
# B2-2：挑幾個詞，把 IDF 印出來對照（這格不用填——直接跑，確認方向）
idf = vectorizer.idf_

for w in ["升息", "台股", "半導體", "毛利率", "原油"]:
    j = list(vocab).index(w)                # 這個詞在詞彙表的第幾格
    value = round(idf[j], 3)                # round(x, 3)：四捨五入到小數第 3 位
    print(w, "IDF =", value)

### B3・查詢向量化 + cosine 排序（核心一刀）

三格：查詢變向量 → 算 cosine → 排名次。

> ⚠️ **全 Lab 最重要的規則：文件用「重新學」的方法、查詢用「照學過的轉換」的方法。** 練 D 還會考一次。

**預期輸出：** 查詢向量 `(1, 23)`；5 篇分數 `[0.628, 0, 0, 0, 0.389]`；文件 0 第 1、文件 4 第 2。

In [ ]:
# B3-1：把查詢也變成向量（必須跟文件在同一個向量空間）
query = "升息"
query_vector = vectorizer.____([query])
# TODO：查詢**不可以**重學一份詞彙表——要「照文件已學過的那份」轉換。B1 用「先學再轉」，這裡該用哪個？
#       ⚠️ 括號坑：它吃「一批文件」，單一查詢也要包成 list（[query]，已幫你包好）——直接餵字串會 ValueError

print("查詢向量形狀：", query_vector.shape, "（1 條查詢 × 詞彙數——跟文件同一空間，才能比相似）")
print()
print("攤開來看（23 個數字，跟 B2-1 的詞彙表一一對應）：")
query_vector.toarray()

### B3-1b・試試看多個字：一次轉三條查詢

上一格說「它吃的是**一批**」——那就真的一次餵三條進去看看。

**預期輸出：** 形狀 `(3, 23)`；第三列（查「黃金」）**23 個格子全是 0**（語料裡沒有「黃金」）。

> 💡 記住這個「全 0」的畫面：等一下 `search()` 要處理的「查無相關文件」就是它造成的。

In [ ]:
query1 = "升息"           # 一個詞
query2 = "升息 台股"      # 兩個詞
query3 = "黃金"           # 語料裡根本沒有的詞

query_vectors = vectorizer.transform([____, ____, ____])   # TODO：一次放三條查詢進 list（用上面三個變數）

print("形狀：", query_vectors.shape, "（3 條查詢 × 詞彙數）")
print()
print("攤開來看（第 3 列是「黃金」——盯著看它有幾個非 0）：")
query_vectors.toarray()

In [ ]:
# B3-2：算 cosine 相似度（查詢 vs 每一篇文件）
from sklearn.metrics.pairwise import ____   # TODO：算「兩向量夾角有多接近」的函式，名字＝餘弦相似度的英文

scores = ____(query_vector, doc_vectors)[0]
# TODO：算「這條查詢」對「每一篇文件」的相似度（同一個函式）
#       ⚠️ 它回 1×5 矩陣，取 [0] 才是那一排分數（已幫你加 [0]）

scores   # 最後一行不寫 print——Jupyter 直接顯示這 5 個分數

In [ ]:
# B3-3：把分數排成名次（不用填，直接跑）
# 先組「(分數, 文件編號)」的 tuple 清單，再 sorted 由大到小
score_list = []
for doc_id, score in enumerate(scores):
    score_list.append((score, doc_id))
ranking = sorted(score_list, reverse=True)     # tuple 排序預設比第一個元素（分數）；reverse=True＝由大到小

print(f"查詢：「{query}」的搜尋結果（依相關度排序）")
print("")
rank = 1
for score, doc_id in ranking:
    if score > 0:
        s = round(score, 3)
        text = docs[doc_id]
        print(f"第{rank}名  分數={s}  文件{doc_id}: {text}")
        rank = rank + 1

> 📊 **加權版回呼：** 開場用「出現**次數**」把文件畫成點；這裡改用 **TF-IDF 加權分數**當座標——同一種圖，只是常見詞被壓小、罕見詞放大，點的位置更準。

In [ ]:
# 📊 把文件畫成「點」：離查詢近＝相關（不用填，直接跑；工具已在最上面載好）
# 23 維畫不出來，挑「升息」「台股」兩個關鍵字當 X、Y 軸（把 23 維壓成看得見的 2 維）
kw_x = "升息"
kw_y = "台股"
vocab = list(vectorizer.get_feature_names_out())
ix = vocab.index(kw_x)      # 「升息」在向量的第幾格
iy = vocab.index(kw_y)      # 「台股」在向量的第幾格

doc_xy = doc_vectors.toarray()                          # 5 篇 × 23，攤成看得見的數字
q_xy = vectorizer.transform(["升息 台股"]).toarray()[0]   # 查詢也變成同空間的向量

plt.figure(figsize=(6, 6))
for doc_id in range(len(docs)):                         # 5 篇文件畫成藍點
    x = doc_xy[doc_id][ix]
    y = doc_xy[doc_id][iy]
    plt.scatter(x, y, s=120, color="steelblue")
    plt.annotate(f"文件{doc_id}", (x, y), xytext=(6, 6), textcoords="offset points")

plt.scatter(q_xy[ix], q_xy[iy], s=320, marker="*", color="crimson")          # 查詢畫成紅星
plt.annotate("查詢：升息 台股", (q_xy[ix], q_xy[iy]), xytext=(8, -16), textcoords="offset points", color="crimson")

plt.xlabel(f"「{kw_x}」分數")
plt.ylabel(f"「{kw_y}」分數")
plt.title("把文件畫成點：離查詢（紅星）近＝相關")
plt.grid(True, alpha=0.3)
plt.show()

# 💡 文件2（原油）、文件3（半導體）在「升息」「台股」兩軸都是 0 → 疊在原點，跟這個查詢無關；
#    文件4（升息+台股都有）離紅星最近 → 最相關，正好對上上一格的排序。

### B4・包成 `search()` 函式（不用填，直接跑）

**跑完看：** 查「半導體 毛利率」→ 文件 3 第 1、文件 1 第 2；查「黃金」→ 正確回「查無」。

**到這裡，你已經做出一個會打分數、排名次的最小搜尋引擎了。**

In [ ]:
def search(query, top_k=3):
    # 💡 跟 B3 完全同一套流程，只是包成函式＋top_k 截斷＋「查無」防呆——之後重複查、換語料都照用
    qv = vectorizer.transform([query])
    scores = cosine_similarity(qv, doc_vectors)[0]
    score_list = []
    for doc_id, score in enumerate(scores):
        score_list.append((score, doc_id))
    ranking = sorted(score_list, reverse=True)

    print("")
    print(f"查詢：「{query}」")
    shown = 0                                  # 已經印出幾名
    for score, doc_id in ranking:
        if score > 0 and shown < top_k:
            shown = shown + 1
            s = round(score, 3)
            text = docs[doc_id]
            print(f"  第{shown}名 分數={s} 文件{doc_id}: {text}")
    if shown == 0:
        print("  （查無相關文件）")

### B4-2・實際用用看（一次一格，感受不同查詢）

函式寫好了，就來查幾次。**盯著第三次**——查「黃金」時走的是「查無相關文件」那條路（就是 B3-1b 那個全 0 向量造成的）。

In [ ]:
____("台股")   # TODO：呼叫剛剛包好的搜尋函式

In [ ]:
____("半導體 毛利率")     # TODO：多詞查詢也行——同一個函式

In [ ]:
____("黃金")              # TODO：故意查語料裡沒有的詞，看它走「查無」那條路

### 📝 小作業 B

1. 用 `search()` 查「**原油 庫存**」——先猜第 1 名是哪篇，再跑。
2. 查「**晶圓代工**」——哪幾篇會出現？為什麼只有它？（想想 B2 的 IDF）
3. 把 `top_k` 調成 5，查「**升息 台股**」——**文件 4**（兩個查詢詞都沾到）和 **文件 0**（只有「升息」但出現 2 次）誰排前面？先猜再跑，然後想：**「兩個查詢詞都命中」和「一個詞重複兩次」，哪個比較能代表這篇跟查詢有關？**

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
search("原油 庫存")           # Q1：文件 2 兩詞都含 → 第 1
search("晶圓代工")            # Q2：只有文件 3 含此罕見詞 → 只撈到它
search("升息 台股", top_k=5)  # Q3：文件 4 排在文件 0 前面 → 多詞覆蓋 > 單詞重複
```
- **Q3 結論：多詞覆蓋 > 單詞重複。**（文件 1 與文件 0 分數非常接近，差在向量長度正規化，超出今天範圍、不影響結論。）
</details>

In [ ]:
# 📝 小作業 B —— 你的答案（做完再看上面摺疊參考解）
# Q1：查「原油 庫存」——先猜第 1 名再跑
# Q2：查「晶圓代工」——想想為什麼只有一篇會出現
# Q3：查「升息 台股」要看到第 5 名，補上控制「印幾名」的參數：
search("升息 台股", ____=5)   # TODO：哪個參數控制「印幾名」？

---
## 🟧 練 C：檢索評估 precision / recall

**漁網類比：** 海裡的魚＝真正相關的文件，你撒網撈起一堆（混著魚和垃圾，海裡還有漏網之魚）。
- **precision** ＝撈起來的那堆裡，**魚佔多少** → 我撈的**準不準**？
- **recall** ＝海裡所有的魚，**被我撈到幾條** → 該撈的**有沒有漏**？

### C1・親手算一次 P / R（兩格：先 precision、再 recall + F1）

**預期輸出：** `Precision = 2/5 = 0.4`　`Recall = 2/4 = 0.5`　`F1 = 0.44`

In [ ]:
# C1-1：precision——「我撈的準不準？」（分母＝我撈起來的全部）
retrieved   = ["d1", "d2", "d3", "d4", "d5"]          # 系統撈回的（依分數排序）
relevant    = {"d1", "d3", "d6", "d7"}                # 標準答案：真正相關的全集（共 4 篇）

hit = []                                   # 撈到的當中，真的相關的（✅魚）
for d in retrieved:
    if d in relevant:
        hit.append(d)

precision = len(____) / len(____)          # TODO：precision＝撈到的魚 ÷ 我撈的全部（分子、分母各放什麼？）

n_hit = len(hit); n_retrieved = len(retrieved)
p = round(precision, 2)
print(f"撈到且相關（✅魚）：{hit}")
print(f"Precision = {n_hit}/{n_retrieved} = {p}")

In [ ]:
# C1-2：recall——「該撈的有沒有漏？」（分母＝海裡所有的魚）＋ F1
recall = len(____) / len(____)             # TODO：recall＝撈到的魚 ÷ 海裡所有的魚（跟上一格比，只有分母換了）

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0
# F1＝precision 與 recall 的調和平均：任一邊掉到 0，F1 就是 0——逼你兩個都顧

n_relevant = len(relevant)
r = round(recall, 2); f1_value = round(f1, 2)
print(f"Recall = {n_hit}/{n_relevant} = {r}")
print(f"F1     = {f1_value}")

### C2・撈多 vs 撈精（不用填，直接跑）

**盯著數字翻轉：** 撈精 → P=**1.0** / R=**0.5**；撈多 → P=**0.5** / R=**1.0**。

> **兩者天生拉扯、沒有兩全，只有依場景取捨。**

In [ ]:
def eval_pr(retrieved, relevant):
    hit = []
    for d in retrieved:
        if d in relevant:
            hit.append(d)
    if len(retrieved) == 0:    # 什麼都沒撈時分母是 0，先擋掉避免除以 0
        p = 0
    else:
        p = len(hit) / len(retrieved)
    r = len(hit) / len(relevant)
    return p, r

relevant = {"d1", "d3", "d6", "d7"}                    # 真正相關共 4 篇
few  = ["d1", "d3"]                                    # (A) 撈精：只撈最有把握的 2 篇
many = ["d1", "d2", "d3", "d4", "d5", "d6", "d7", "d8"] # (B) 撈多：疑似相關全撈，8 篇

for name, r_list in [("撈精(2篇)", few), ("撈多(8篇)", many)]:
    p, r = eval_pr(r_list, relevant)
    print(f"{name}: Precision={round(p,2)}  Recall={round(r,2)}")

### 📝 小作業 C

1. 系統這次只回 **top-3**（`["d1", "d2", "d3"]`，relevant 不變）——用 `eval_pr()` 算 P/R。跟 C1「撈 5 篇」比，**哪個指標動了、為什麼？**
2. **紙筆討論題：** 某銀行做「法遵／反洗錢檢索系統」，撈出**所有可能相關**的可疑交易紀錄與適用法規供查核。設計時**最該優先確保**哪個指標？
   - (A) precision　(B) recall，因為漏掉一筆關鍵可疑交易代價極大　(C) 速度最重要　(D) precision，因為沒時間看太多

> 🆕 **法遵（Compliance）** ＝確認公司做的事都符合法規；「**可疑交易**」＝像洗錢那類異常金流，依法必須找出來通報。

<details><summary>📖 做完再看：參考解</summary>

```python
p, r = eval_pr(["d1", "d2", "d3"], relevant)   # top-3: Precision=0.67 Recall=0.5
```
- **Q1：動的是 precision（升到 0.67），recall 不動（0.5）。** 撈得少 → precision 分母變小。
- **Q2：(B) recall。** 法遵場景「漏掉」的代價最大 → 重 recall。**考點：由場景的「錯誤代價」決定重誰。**
</details>

In [ ]:
# 📝 小作業 C —— 你的答案（做完再看上面摺疊參考解）
# Q1：把 retrieved 換成只回 top-3 的清單、relevant 不變，用 eval_pr 算 P/R，跟 C1 撈 5 篇比哪個指標動了
# p, r = eval_pr(____, ____)   # TODO：兩個參數各放什麼？（retrieved / relevant）
# Q2：先想理由，再回上面選 (A)~(D)

---
## 🟫 練 D：讀真實中文新聞檔 + 整合搜尋

把寫死的 5 句換成 `data/news/` 裡的 **15 篇真實中文新聞**。

> ⚠️ **真實中文沒有空白**——直接餵會把整句當一個詞，查什麼都比不到。**中文檢索第一步＝先斷詞**（用 `jieba`）。

先跑下面的 jieba 檢查格；裝不起來也別慌——小作業 D 的 ⭐ 有不用 jieba 的 Backup。

In [ ]:
# ✅ 練 D 環境檢查：中文斷詞需要 jieba（練 A–C 用不到，所以放到這裡才檢查）
try:
    import jieba
    print("jieba OK，版本：", jieba.__version__)
except ModuleNotFoundError:
    print("❌ 沒裝 jieba → 終端機執行：pip install jieba")
    print("   真的裝不起來也別慌——小作業 D 的 ⭐ 有「完全不用 jieba」的 char n-gram Backup。")

### D1-1a・先看一眼：jieba 直接斷詞（**還沒加詞典**）

中文沒有空白，得靠斷詞器切。先讓 jieba **原封不動**斷一句金融新聞看看。

**預期輸出：** 一串詞。**盯著兩個地方**——`晶圓代工` 被切成 `晶圓/代工`、`監理沙盒` 被切成 `監理/沙盒`。

> ⚠️ 專有名詞被切散＝搜尋時就比不到了。下一格解決它。

In [ ]:
# D1-1a：jieba 原味斷詞（還沒加任何詞典）
import glob, os, jieba

sentence = "晶圓代工報價上揚，監理沙盒開放電動車保險試辦"

jieba.____(sentence)   # TODO：斷詞後直接回「一個詞的清單」的方法（提示：l 代表 list）
# 盯著看：「晶圓代工」「監理沙盒」會被切散
# 最後一行不寫 print——Jupyter 會直接顯示

### D1-1b・加上自訂詞典，**同一句再斷一次**

把金融專有名詞先「釘」成一個詞，再斷同一句。

**預期輸出：** `晶圓代工` 和 `監理沙盒` 這次**各自是一個完整的詞**了。

> 這就是自訂詞典的價值——**兩格對照著看，差別一目了然**。

In [ ]:
# D1-1b：把專有名詞加進詞典，再斷同一句
for w in ["晶圓代工", "毛利率", "半導體", "升息", "央行", "監理沙盒", "電動車", "再生能源", "每股盈餘"]:
    jieba.____(w)   # TODO：把這個詞「釘」進詞典（斷詞時不再被拆開）——jieba 的哪個方法？
# 詞多時可寫成一個檔，用 jieba.load_userdict("fin_dict.txt") 一次載入

jieba.____(sentence)   # TODO：同一句、同一個斷詞方法（跟上一格 D1-1a 一樣）
# 對照上一格：「晶圓代工」「監理沙盒」現在各自完整了
# 最後一行不寫 print——Jupyter 會直接顯示

### D1-1c・先認識 `str` 的 `join`：把一串詞黏回一個字串

上一格斷出來是「一串詞（list）」，但 `TfidfVectorizer` 要的是「用空白隔開的字串」。`join` 就是把清單黏成字串的工具。

**預期輸出：** `央行 決議 升息`（用空白黏）／`央行／決議／升息`（換個字元黏）。

In [ ]:
# str 的 join：把「一串詞」用某個字元黏成「一個字串」
words = ["央行", "決議", "升息"]
print(" ".join(words))     # 央行 決議 升息   ← 用空白黏（下一格 cut() 就用這招）
print("／".join(words))    # 央行／決議／升息  ← 換個字元黏，join 前面放什麼就用什麼黏

In [ ]:
# D1-1c：包成 cut()——斷詞後用空白接回成一個字串
def cut(text):
    return " ".____(jieba.lcut(text.strip()))  # TODO：把「一串詞」用空白黏成一個字串——上一格剛學的方法
#        （黏成空白分隔的字串，才餵得進 token_pattern=\S+）

print(cut("央行決議升息半碼，市場解讀偏鷹"))   # 看到詞與詞之間被空白隔開就對了

In [ ]:
# D1-2a：glob 讀 data/news/*.txt（讀進來先斷詞）
paths = sorted(glob.glob("data/news/*.txt"))
docs = []
names = []
for p in paths:
    with ____(p, encoding="utf-8") as f:       # TODO：開檔讀取的內建函式（記得 encoding="utf-8"）
        docs.append(cut(f.read()))             # ← 關鍵：讀進來「先斷詞」再放進語料
        names.append(os.path.basename(p))

print(f"讀到 {len(docs)} 篇新聞：{names[:3]} ...")

### D1-2b・⚠️ 換語料了，該用哪一個？（**故意先錯一次**）

語料從「5 句短句」換成「15 篇新聞」了。這裡**故意先用錯的那個**，看會發生什麼事。

**預期輸出：** 形狀 `(15, 23)`。

> 🔍 **盯住第二個數字：23。** 那是**舊那 5 句短句**的詞彙表大小——新語料被硬塞進舊的 23 個格子裡。**錯了會當場現形。**

In [ ]:
# D1-2b：故意用「只轉、不學」的那個——看它壞在哪
doc_vectors = vectorizer.____(docs)   # TODO：先故意用「只轉、不學」的那個（就是查詢用的那個方法）

print("向量矩陣形狀：", doc_vectors.shape)
print("⚠️ 第二個數字是 23 嗎？那是舊語料（5 句短句）的詞彙表大小——新語料的詞根本沒被學進去")

### D1-2c・改用正確的那個

**預期輸出：** 形狀 `(15, 320)`。

> 🔑 **23 → 320**：詞彙表真的重學過了。**規則再說一次：換語料 ＝ 一定要重新「學」一次。**

In [ ]:
# D1-2c：換成正確的——重新學一份詞彙表
doc_vectors = vectorizer.____(docs)   # TODO：換語料一定要「重新學」——B1 文件用的那個方法

print("向量矩陣形狀：", doc_vectors.shape)
print("✅ 第二個數字變成三位數了嗎？代表 15 篇新聞的詞彙表真的重學過了")

### D2・整合成 `search_files()`：一條龍搜尋

把所有零件接成一個能搜「整個資料夾」的小引擎。

> ⚠️ **查詢也要先斷詞**——文件走了 `cut()`、查詢沒走，就對不上同一份詞彙表。

**預期輸出：** 查「升息」→ `001_升息.txt` 第 1；查「台股 半導體」→ `002`／`003` 前兩名。

In [ ]:
def search_files(query, top_k=3):
    qv = vectorizer.transform([cut(query)])    # ⚠️ 查詢也要先斷詞！（和文件用同一套 cut）
    scores = cosine_similarity(qv, doc_vectors)[0]
    score_list = []
    for doc_id, score in enumerate(scores):
        score_list.append((score, doc_id))
    ranking = sorted(score_list, reverse=True)

    print("")
    print(f"查詢：「{query}」")
    shown = 0
    for score, doc_id in ranking:
        if score > 0 and shown < top_k:
            shown = shown + 1
            s = round(score, 3)
            name = names[doc_id]
            snippet = docs[doc_id][:24]
            print(f"  第{shown}名 分數={s} 【{name}】{snippet}...")

In [ ]:
____("升息")   # TODO：呼叫剛包好的「搜整個資料夾」函式

In [ ]:
____("台股 半導體")   # TODO：同一個函式，多詞查詢

### 📝 小作業 D

1. 用 `search_files()` 查「**電動車**」——第 1 名是不是 `005_電動車.txt`？
2. **觀察題：** 查「**綠能 政策**」——第 1 名**很可能不是** `015_綠能.txt`。打開 `data/news/015_綠能.txt` 看原文，想想為什麼？（提示：015 用的是哪些詞？查詢真正比中的是哪個？）
3. **⭐ 進階（選做）：不用 jieba 的 Backup。** 用 `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))`（字的 2~3 連字組合當特徵，免斷詞）重建索引查「升息」。想一想：char_wb 吃**沒斷詞的原文**，你手上 `docs` 還能用嗎？（它已被 `cut()` 過）

<details><summary>📖 做完再看：參考解</summary>

- **Q2（重點）：** 015 原文寫的是「**再生能源**」「綠電」「太陽能」——**字面幾乎沒有「綠能」**！查詢真正比中的只剩「政策」，而 005/004 也含「政策」→ 015 排不到第 1。
  - 🔑 **這就是 TF-IDF 關鍵字比對的天花板：** 「綠能」和「再生能源」意思一樣、字面不同就比不到。**下午的 embedding 正是為解決這件事——下午會回收。**
</details>

In [ ]:
# 📝 小作業 D —— 你的答案（做完再看上面摺疊參考解）
# search_files("電動車")      # Q1
# search_files("綠能 政策")   # Q2：跑完打開 015 原文，想想為什麼第 1 名不是它

In [ ]:
# ⭐ 小作業 D 進階（選做）—— 你的答案（做完再看完整版）
# 提示：char_wb 吃「沒斷詞的原文」，docs 已被 cut() 過，要用 open() 重讀一次原始檔到 raw_docs，
#       再 TfidfVectorizer(analyzer="char_wb", ngram_range=(2,3)) 建索引查「升息」。

---
## 🚀 附錄（選做）：BM25 排序對比

> 需先 `pip install rank_bm25`；卡在前面的直接跳過，不影響核心。

業界排序實際多用 TF-IDF 的升級版 **BM25**（`Elasticsearch` 預設就是它）。

**看什麼：** BM25 分數**不是 0~1**、跟 cosine 是兩套尺度，但**排序方向一樣**——查「升息」仍是文件 0 第 1、文件 4 第 2。

In [ ]:
# BM25 附錄用回練 A/B 的 5 篇短句語料（練 D 已把 docs 換成 15 篇新聞，這裡換回來對照 B3 排序）
docs = [
    "升息 抑制 通膨 央行 升息 一碼",          # 文件0：升息
    "台股 開高 半導體 領漲 台股 收紅",        # 文件1：台股/半導體
    "原油 庫存 下滑 油價 走高",               # 文件2：原油
    "半導體 毛利率 提升 晶圓代工 報價 上揚",   # 文件3：半導體財報
    "升息 衝擊 台股 資金 外流",               # 文件4：升息+台股
]
print("已把 docs 換回 5 篇短句語料")

In [ ]:
from rank_bm25 import BM25Okapi   # BM25Okapi＝rank_bm25 提供的 BM25 實作類別（名字記不住沒關係，直接用）

tokenized_docs = []
for d in docs:
    tokenized_docs.append(d.split())   # BM25 吃「已斷詞的 list」——每篇是一串詞
bm25 = BM25Okapi(tokenized_docs)

q = "升息".split()
bm25_scores = bm25.get_scores(q)
# 💡 BM25 分數不是 0~1（跟 cosine 是兩套尺度）——只看排序方向、不對數字

score_list = []
for doc_id, score in enumerate(bm25_scores):
    score_list.append((score, doc_id))
ranking = sorted(score_list, reverse=True)

print("BM25 排序（查「升息」）：")
rank = 1
for score, doc_id in ranking:
    if score > 0:
        print(f"第{rank}名  BM25分數={round(score,3)}  文件{doc_id}: {docs[doc_id]}")
        rank = rank + 1

---
## 🗺 總結：你今天親手做了一個古典搜尋引擎

| 練 | 你做了什麼 | 在搜尋引擎裡的角色 |
|---|---|---|
| A | `dict` 手刻倒排索引 + 布林 AND/OR | **秒查**哪些文件含這個詞 |
| B | `TfidfVectorizer` + cosine + `search()` | **排序**——回答「多相關」 |
| C | 手算 P/R + `eval_pr()` | **評估**——準不準、有沒有漏 |
| D | jieba 斷詞 + 讀 15 篇真實中文新聞 | **真實語料**——斷詞品質決定檢索品質 |

**🔗 接下午：** 你做的是「把文件變向量、用 cosine 比相似」——但 TF-IDF 比的是**字面**：搜「升息」找不到寫「調升利率」的文件。下午把 `TfidfVectorizer` 換成 **embedding 模型**，同一套骨架就升級成**語意檢索**——**Lab 3 的 RAG，「檢索」那一半用的正是這個流程。**